In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
import heapq
import itertools
import time
from enum import IntEnum

class Actions(IntEnum):
    HOLD = 0
    BUY = 1
    SELL = 2

# State Vector: [5 Bids, 5 Asks, Inventory, Cash, Spread, Volatility]
OBSERVATION_DIMS = 14

# Normalization Constants
INITIAL_CASH = 100_000.0
MAX_STEPS_PER_EPISODE = 1000
MAX_INVENTORY_LIMIT = 100
MAX_CASH_LIMIT = 1_000_000
TRANSACTION_COST = 1.0  # Cost per trade to prevent churning


class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  # Max-Heap (negative prices)
        self.asks = []  # Min-Heap (positive prices)
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                # Negate price for max-heap behavior in Python
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            # Execution
            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side,
                'aggressor': 'market' if limit_price == float('inf') else 'limit'
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# 3. THE GYM ENVIRONMENT (TradingEnv)

class TradingEnv(gym.Env):
    """
    The 'Video Game' wrapper for the Agent.
    """
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super(TradingEnv, self).__init__()
        
        # 1. Define Action Space: Discrete(3) -> [0, 1, 2]
        self.action_space = spaces.Discrete(len(Actions))
        
        # 2. Define Observation Space: Continuous Vector(14)
        # We use float32 for Neural Network compatibility
        self.observation_space = spaces.Box(
            low=-np.inf, 
            high=np.inf, 
            shape=(OBSERVATION_DIMS,), 
            dtype=np.float32
        )
        
        # Internal State
        self.engine = None
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price